In [1]:
import pandas as pd
df = pd.read_parquet("deja_fn_clustered.parquet")

print(df.shape)
print(df["cluster_id"].value_counts())

(1404, 1545)
cluster_id
5    738
4    207
3    126
2    116
1    116
0    101
Name: count, dtype: int64


In [2]:
df.groupby("cluster_id")["margin"].apply(lambda s: s.abs().mean()).sort_values(ascending=False).head(20)

cluster_id
0    0.005730
1    0.003633
2    0.003473
4    0.002887
5    0.002885
3    0.002351
Name: margin, dtype: float32

In [3]:
import pandas as pd
import numpy as np
import re
import unicodedata
import difflib
from collections import Counter

df = pd.read_parquet("deja_fn_clustered.parquet")

# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def normalize(s):
    return unicodedata.normalize("NFKC", str(s)).strip()

def is_ascii(c):
    return ord(c) < 128

def has_digit(c):
    return c.isdigit()

def is_separator(c):
    return c in "-_."

def levenshtein(a, b):
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            cur = dp[j]
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + cost)
            prev = cur
    return dp[m]

# ---------------------------------------------------------
# Mechanism detection
# ---------------------------------------------------------

def classify_pair(a_raw, b_raw):
    a = normalize(a_raw)
    b = normalize(b_raw)

    result = {}

    # Basic properties
    result["punycode_attack"] = ("xn--" in a) or ("xn--" in b)
    result["separator_change"] = (("-" in a) ^ ("-" in b)) or \
                                 (("_" in a) ^ ("_" in b)) or \
                                 (("." in a) ^ ("." in b))

    # Edit analysis via SequenceMatcher
    sm = difflib.SequenceMatcher(a=a, b=b)
    subs = 0
    inserts = 0
    deletes = 0
    transpositions = 0
    digit_sub = False
    unicode_sub = False

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "replace":
            sa = a[i1:i2]
            sb = b[j1:j2]
            if len(sa) == len(sb):
                for ca, cb in zip(sa, sb):
                    if ca != cb:
                        subs += 1
                        if has_digit(ca) or has_digit(cb):
                            digit_sub = True
                        if not is_ascii(ca) or not is_ascii(cb):
                            unicode_sub = True
            else:
                subs += max(len(sa), len(sb))

        elif tag == "insert":
            inserts += len(b[j1:j2])
        elif tag == "delete":
            deletes += len(a[i1:i2])

    # Transposition heuristic
    if len(a) == len(b):
        diff_positions = [i for i in range(len(a)) if a[i] != b[i]]
        if len(diff_positions) == 2:
            i, j = diff_positions
            if a[i] == b[j] and a[j] == b[i]:
                transpositions = 1

    # Extension detection
    extension = (a.startswith(b) or b.startswith(a)) and abs(len(a) - len(b)) > 2

    result["has_substitution"] = subs > 0
    result["has_insertion"] = inserts > 0
    result["has_deletion"] = deletes > 0
    result["has_transposition"] = transpositions > 0
    result["has_extension"] = extension
    result["digit_substitution"] = digit_sub
    result["unicode_homoglyph"] = unicode_sub

    # ---------------------------------------------------------
    # Primary mechanism (priority order)
    # ---------------------------------------------------------
    if result["punycode_attack"]:
        result["primary_mechanism"] = "punycode"
    elif result["digit_substitution"]:
        result["primary_mechanism"] = "digit_substitution"
    elif result["unicode_homoglyph"]:
        result["primary_mechanism"] = "unicode_homoglyph"
    elif result["has_transposition"]:
        result["primary_mechanism"] = "transposition"
    elif result["has_extension"]:
        result["primary_mechanism"] = "extension"
    elif result["has_insertion"] and not result["has_deletion"]:
        result["primary_mechanism"] = "insertion"
    elif result["has_deletion"] and not result["has_insertion"]:
        result["primary_mechanism"] = "deletion"
    elif result["has_substitution"]:
        result["primary_mechanism"] = "substitution"
    else:
        result["primary_mechanism"] = "other"

    return result

# ---------------------------------------------------------
# Apply classification
# ---------------------------------------------------------

mechanisms = df.apply(
    lambda row: classify_pair(row["fraudulent_name"], row["real_name"]),
    axis=1,
    result_type="expand"
)

df = pd.concat([df, mechanisms], axis=1)

# ---------------------------------------------------------
# Inspect distribution
# ---------------------------------------------------------

print("\nPrimary mechanism distribution:")
print(df["primary_mechanism"].value_counts())

print("\nMechanism by cluster:")
print(pd.crosstab(df["cluster_id"], df["primary_mechanism"], normalize="index"))


Primary mechanism distribution:
primary_mechanism
deletion              480
substitution          299
unicode_homoglyph     248
other                 192
digit_substitution    123
insertion              39
extension              17
transposition           6
Name: count, dtype: int64

Mechanism by cluster:
primary_mechanism  deletion  digit_substitution  extension  insertion  \
cluster_id                                                              
0                  0.000000            0.000000   0.000000   0.000000   
1                  0.586207            0.034483   0.017241   0.008621   
2                  0.456897            0.034483   0.008621   0.008621   
3                  0.079365            0.079365   0.000000   0.103175   
4                  0.231884            0.096618   0.004831   0.033816   
5                  0.407859            0.115176   0.017615   0.023035   

primary_mechanism     other  substitution  transposition  unicode_homoglyph  
cluster_id                   

In [14]:
df.groupby("cluster_id")[["delta_norm","margin"]].corr()

delta_norm    margin
cluster_id                                 
0          delta_norm         NaN       NaN
           margin             NaN  1.000000
1          delta_norm    1.000000 -0.421599
           margin       -0.421599  1.000000
2          delta_norm    1.000000 -0.495856
           margin       -0.495856  1.000000
3          delta_norm    1.000000 -0.368342
           margin       -0.368342  1.000000
4          delta_norm    1.000000 -0.414941
           margin       -0.414941  1.000000
5          delta_norm    1.000000 -0.285419
           margin       -0.285419  1.000000